<a href="https://colab.research.google.com/github/sakaleshhubli/sem7-ML2-/blob/main/6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title 📊 Ensemble Classifier Comparison Dashboard {display-mode: "form"}

import json
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from IPython.display import HTML, display
from google.colab import output

# 1. Register JavaScript Error Callback
def _report_js_error(message):
    print(f"JavaScript Error: {message}")
output.register_callback('report_js_error', _report_js_error)

# 2. Load Dataset and Prepare Data
data = load_wine()
X, y = data.data, data.target
feature_names = list(data.feature_names)
target_names = list(data.target_names)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 3. Train Bagging Models
bagging_base = BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=50, random_state=42)
bagging_base.fit(X_train, y_train)
y_pred_bag = bagging_base.predict(X_test)

rf_model = RandomForestClassifier(n_estimators=50, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

# 4. Train Boosting Models
adaboost_model = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1), n_estimators=50, random_state=42, algorithm='SAMME')
adaboost_model.fit(X_train, y_train)
y_pred_ada = adaboost_model.predict(X_test)

gb_model = GradientBoostingClassifier(n_estimators=50, random_state=42)
gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)

# 5. Evaluate Performance
def evaluate(y_true, y_pred):
    return {
        "accuracy": round(accuracy_score(y_true, y_pred), 3),
        "precision": round(precision_score(y_true, y_pred, average='weighted'), 3),
        "recall": round(recall_score(y_true, y_pred, average='weighted'), 3),
        "f1": round(f1_score(y_true, y_pred, average='weighted'), 3)
    }

results = {
    "Bagging (Decision Trees)": evaluate(y_test, y_pred_bag),
    "Random Forest (Bagging)": evaluate(y_test, y_pred_rf),
    "AdaBoost (Boosting)": evaluate(y_test, y_pred_ada),
    "Gradient Boosting (Boosting)": evaluate(y_test, y_pred_gb)
}

# Extract feature importances from RF
rf_importances = rf_model.feature_importances_
features_sorted_indices = np.argsort(rf_importances)[::-1]
feature_importance_data = [
    {"feature": feature_names[idx], "importance": round(float(rf_importances[idx]), 4)}
    for idx in features_sorted_indices
]

# Prepare payload for dashboard
payload = {
    "dataset_info": {
        "name": "Wine Classification Dataset",
        "samples": X.shape[0],
        "features": X.shape[1],
        "classes": len(target_names)
    },
    "metrics": results,
    "feature_importances": feature_importance_data
}

# 6. Render Dashboard
html_template = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <style>
        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background-color: #f4f6f8;
            color: #333;
            margin: 0;
            padding: 20px;
        }
        .dashboard-container {
            max-width: 1200px;
            margin: 0 auto;
        }
        .header {
            margin-bottom: 24px;
        }
        .header h1 {
            margin: 0;
            font-size: 24px;
            color: #2c3e50;
        }
        .header p {
            margin: 4px 0 0 0;
            color: #7f8c8d;
            font-size: 14px;
        }
        .kpi-row {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(220px, 1fr));
            gap: 16px;
            margin-bottom: 24px;
        }
        .card {
            background-color: #ffffff;
            border-radius: 8px;
            padding: 20px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.05);
            border: 1px solid #eef2f5;
        }
        .kpi-card {
            text-align: center;
        }
        .kpi-card .value {
            font-size: 28px;
            font-weight: bold;
            color: #3498db;
            margin-bottom: 4px;
        }
        .kpi-card .label {
            font-size: 12px;
            color: #95a5a6;
            text-transform: uppercase;
            letter-spacing: 0.5px;
        }
        .main-layout {
            display: grid;
            grid-template-columns: 2fr 1fr;
            gap: 20px;
            margin-bottom: 24px;
        }
        @media (max-width: 900px) {
            .main-layout {
                grid-template-columns: 1fr;
            }
        }
        .chart-card {
            display: flex;
            flex-direction: column;
        }
        .chart-card h2 {
            font-size: 16px;
            color: #2c3e50;
            margin-top: 0;
            margin-bottom: 16px;
            border-bottom: 1px solid #eee;
            padding-bottom: 8px;
        }
        .canvas-wrapper {
            position: relative;
            flex-grow: 1;
            min-height: 280px;
            max-height: 400px;
        }
        .table-section {
            margin-top: 24px;
        }
        table {
            width: 100%;
            border-collapse: collapse;
            background: white;
            border-radius: 8px;
            overflow: hidden;
            box-shadow: 0 2px 4px rgba(0,0,0,0.05);
        }
        th, td {
            padding: 12px 15px;
            text-align: left;
            border-bottom: 1px solid #f1f1f1;
        }
        th {
            background-color: #f8f9fa;
            font-weight: 600;
            color: #2c3e50;
        }
        tr:last-child td {
            border-bottom: none;
        }
        .badge {
            padding: 4px 8px;
            border-radius: 4px;
            font-size: 11px;
            font-weight: bold;
            text-transform: uppercase;
        }
        .bagging {
            background-color: #e3f2fd;
            color: #0d47a1;
        }
        .boosting {
            background-color: #efebe9;
            color: #4e342e;
        }
    </style>
    <!-- Load Chart.js via CDN -->
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
</head>
<body>
    <div class="dashboard-container">
        <div class="header">
            <h1>Ensemble Techniques Comparison</h1>
            <p id="dataset-subtitle">Evaluating Bagging and Boosting algorithms</p>
        </div>

        <!-- KPI Cards -->
        <div class="kpi-row">
            <div class="card kpi-card">
                <div class="value" id="kpi-dataset">Wine</div>
                <div class="label">Dataset</div>
            </div>
            <div class="card kpi-card">
                <div class="value" id="kpi-features">-</div>
                <div class="label">Features</div>
            </div>
            <div class="card kpi-card">
                <div class="value" id="kpi-best-model">-</div>
                <div class="label">Top Performer</div>
            </div>
            <div class="card kpi-card">
                <div class="value" id="kpi-max-accuracy">-</div>
                <div class="label">Peak Accuracy</div>
            </div>
        </div>

        <!-- Main Charts Section -->
        <div class="main-layout">
            <div class="card chart-card">
                <h2>Model Metrics Benchmark</h2>
                <div class="canvas-wrapper">
                    <canvas id="comparisonChart"></canvas>
                </div>
            </div>
            <div class="card chart-card">
                <h2>Feature Importances (Random Forest)</h2>
                <div class="canvas-wrapper">
                    <canvas id="importanceChart"></canvas>
                </div>
            </div>
        </div>

        <!-- Comprehensive Performance Table -->
        <div class="table-section">
            <div class="card">
                <h2 style="font-size: 16px; margin-top: 0; margin-bottom: 16px; color: #2c3e50;">Detailed Performance Report</h2>
                <div style="overflow-x: auto;">
                    <table id="performance-table">
                        <thead>
                            <tr>
                                <th>Model Name</th>
                                <th>Type</th>
                                <th>Accuracy</th>
                                <th>Precision (Weighted)</th>
                                <th>Recall (Weighted)</th>
                                <th>F1-Score (Weighted)</th>
                            </tr>
                        </thead>
                        <tbody>
                        </tbody>
                    </table>
                </div>
            </div>
        </div>
    </div>

    <script>
        window.onerror = function(message) {
            google.colab.kernel.invokeFunction('report_js_error', [message], {});
        };

        const data = DATA_PLACEHOLDER;

        // Set KPIs
        document.getElementById('kpi-dataset').innerText = data.dataset_info.name.split(' ')[0];
        document.getElementById('kpi-features').innerText = data.dataset_info.features;
        document.getElementById('dataset-subtitle').innerText = `Dataset: ${data.dataset_info.name} | Samples: ${data.dataset_info.samples} | Classes: ${data.dataset_info.classes}`;

        let bestModel = '';
        let maxAcc = -1;

        // Populate Table and compute KPIs
        const tbody = document.querySelector('#performance-table tbody');
        Object.keys(data.metrics).forEach(model => {
            const metrics = data.metrics[model];
            if (metrics.accuracy > maxAcc) {
                maxAcc = metrics.accuracy;
                bestModel = model;
            }

            const isBagging = model.includes('Bagging') || model.includes('Forest');
            const badgeClass = isBagging ? 'bagging' : 'boosting';
            const badgeText = isBagging ? 'Bagging' : 'Boosting';

            const row = document.createElement('tr');
            row.innerHTML = `
                <td><strong>${model}</strong></td>
                <td><span class="badge ${badgeClass}">${badgeText}</span></td>
                <td>${(metrics.accuracy * 100).toFixed(1)}%</td>
                <td>${metrics.precision.toFixed(3)}</td>
                <td>${metrics.recall.toFixed(3)}</td>
                <td>${metrics.f1.toFixed(3)}</td>
            `;
            tbody.appendChild(row);
        });

        document.getElementById('kpi-best-model').innerText = bestModel.split(' ')[0];
        document.getElementById('kpi-max-accuracy').innerText = (maxAcc * 100).toFixed(1) + '%';

        // Comparison Bar Chart
        const models = Object.keys(data.metrics);
        const accuracies = models.map(m => data.metrics[m].accuracy);
        const precisions = models.map(m => data.metrics[m].precision);
        const recalls = models.map(m => data.metrics[m].recall);
        const f1s = models.map(m => data.metrics[m].f1);

        const ctxComp = document.getElementById('comparisonChart').getContext('2d');
        new Chart(ctxComp, {
            type: 'bar',
            data: {
                labels: models,
                datasets: [
                    { label: 'Accuracy', data: accuracies, backgroundColor: '#3498db' },
                    { label: 'Precision', data: precisions, backgroundColor: '#2ecc71' },
                    { label: 'Recall', data: recalls, backgroundColor: '#e67e22' },
                    { label: 'F1-Score', data: f1s, backgroundColor: '#9b59b6' }
                ]
            },
            options: {
                responsive: true,
                maintainAspectRatio: false,
                scales: {
                    y: { min: 0.8, max: 1.0, ticks: { format: { style: 'percent' } } }
                }
            }
        });

        // Feature Importance Chart
        const topFeatures = data.feature_importances.slice(0, 5);
        const ctxImp = document.getElementById('importanceChart').getContext('2d');
        new Chart(ctxImp, {
            type: 'doughnut',
            data: {
                labels: topFeatures.map(f => f.feature),
                datasets: [{
                    data: topFeatures.map(f => f.importance),
                    backgroundColor: ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db']
                }]
            },
            options: {
                responsive: true,
                maintainAspectRatio: false,
                plugins: {
                    legend: { position: 'bottom' }
                }
            }
        });
    </script>
</body>
</html>
""".replace("DATA_PLACEHOLDER", json.dumps(payload))

display(HTML(html_template))


/usr/local/lib/python3.13/dist-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(


Model Name,Type,Accuracy,Precision (Weighted),Recall (Weighted),F1-Score (Weighted)
